# Lab 2: MNIST with a Neural Network (MLP)
AI Lab, University of Technology Bahrain. About 10 minutes. GPU recommended (CPU works).

**You will learn**
- Building a multilayer perceptron in Keras
- Softmax outputs, cross-entropy loss and the Adam optimizer
- Train/validation curves and spotting overfitting

**Before you run**
- On Kaggle: right-hand panel, *Settings → Accelerator → GPU T4 x2* (when the lab needs a GPU) and *Settings → Internet → On*. GPU and Internet need a phone-verified account.
- On Colab: *Runtime → Change runtime type → T4 GPU*.
- Run the cells from top to bottom with **Shift + Enter**.

In [ ]:
# Setup: run this cell first
import os, sys, glob, time
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

ON_KAGGLE = os.path.exists("/kaggle/working")
ON_COLAB = "google.colab" in sys.modules
OUT_DIR = "/kaggle/working" if ON_KAGGLE else "/content" if ON_COLAB else "."
gpus = tf.config.list_physical_devices("GPU")
print("Platform:", "Kaggle" if ON_KAGGLE else "Colab" if ON_COLAB else "local")
print("TensorFlow", tf.__version__, "| Keras", keras.__version__)
print("GPU:", [g.name for g in gpus] if gpus else "none, running on CPU. See 'Before you run' above to switch a GPU on")
keras.utils.set_random_seed(42)

In [ ]:
# Dataset loader: downloads with Keras; on Kaggle it can also use a copy attached with "+ Add Input"
def load_dataset(name):
    """name: 'mnist', 'fashion_mnist' or 'cifar10' -> (x_train, y_train), (x_test, y_test)"""
    try:
        return getattr(keras.datasets, name).load_data()
    except Exception as err:
        if name == "mnist":  # Kaggle-attached copy of mnist.npz, if any
            hits = glob.glob("/kaggle/input/**/mnist.npz", recursive=True)
            if hits:
                d = np.load(hits[0])
                return (d["x_train"], d["y_train"]), (d["x_test"], d["y_test"])
        raise RuntimeError(
            "The dataset did not download. On Kaggle, switch on Settings > Internet "
            "(needs a phone-verified account). On Colab, check your connection. Then run this cell again.") from err

In [ ]:
(x_train, y_train), (x_test, y_test) = load_dataset("mnist")
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0
print(x_train.shape, y_train[:10])

## 1. Build the network
`Flatten` turns each 28×28 image into 784 inputs → two hidden layers → 10 outputs (one probability per digit).

In [ ]:
model = keras.Sequential([
    keras.Input(shape=(28, 28)),
    layers.Flatten(),
    layers.Dense(256, activation="relu"),
    layers.Dropout(0.2),
    layers.Dense(128, activation="relu"),
    layers.Dense(10, activation="softmax"),
])
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()

## 2. Train

In [ ]:
t = time.time()
history = model.fit(x_train, y_train, epochs=8, batch_size=128, validation_split=0.1, verbose=2)
print(f"Training time: {time.time()-t:.1f}s")

In [ ]:
# Learning curves
def plot_history(h):
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
    ax[0].plot(h.history["loss"], label="train"); ax[0].plot(h.history["val_loss"], label="validation")
    ax[0].set_title("Loss"); ax[0].set_xlabel("epoch"); ax[0].legend()
    ax[1].plot(h.history["accuracy"], label="train"); ax[1].plot(h.history["val_accuracy"], label="validation")
    ax[1].set_title("Accuracy"); ax[1].set_xlabel("epoch"); ax[1].legend()
    plt.tight_layout(); plt.show()
plot_history(history)

## 3. Evaluate and inspect mistakes

In [ ]:
loss, acc = model.evaluate(x_test, y_test, verbose=0)
print(f"Test accuracy: {acc:.4f}")

probs = model.predict(x_test, verbose=0)
pred = probs.argmax(1)
wrong = np.where(pred != y_test)[0][:12]
fig, axes = plt.subplots(1, 12, figsize=(14, 1.9))
for ax, i in zip(axes, wrong):
    ax.imshow(x_test[i], cmap="gray"); ax.set_title(f"{y_test[i]}→{pred[i]}", color="crimson"); ax.axis("off")
plt.suptitle("Mistakes (true → predicted)"); plt.show()

In [ ]:
model.save(f"{OUT_DIR}/mnist_mlp.keras")
print("Saved to", f"{OUT_DIR}/mnist_mlp.keras")

## Try it
1. Remove the `Dropout` layer and train for 20 epochs. Do the train and validation curves split apart?
2. Try `optimizer="sgd"`. How does the learning speed compare?
3. Run once with a GPU and once without, and compare training time. Why is the gap small for this model?

### Save your work
- **Kaggle:** *Save Version → Quick Save* keeps your edits; *Save & Run All* also keeps the outputs. Files written to `/kaggle/working` appear under **Output**.
- **Colab:** *File → Save a copy in Drive*. Files in `/content` are deleted when the session ends, so download anything you need from the Files panel first.